In [ ]:
# ============================================================
# AKILLI MÜŞTERİ YÖNETİM VE ANALİZ SİSTEMİ
# TELCO / TELEKOMÜNİKASYON SENARYOSU
# ============================================================
# Bu proje, bir telekom şirketindeki müşterileri analiz eder.
# Kullanıcıdan alınan bilgilerle müşteri tipi, fatura durumu,
# churn riski ve kampanya önerisi hesaplanır.
#
# Churn riski: Müşterinin şirketten ayrılma ihtimalidir.
# ============================================================

import random
import math
from datetime import date


# ------------------------------------------------------------
# 1. PROGRAM BAŞLIĞI
# ------------------------------------------------------------

print("AKILLI MÜŞTERİ YÖNETİM VE ANALİZ SİSTEMİ")
print("Telco Senaryosu - Kullanıcı Girişli Model")
print("-" * 60)


# ------------------------------------------------------------
# 2. KULLANICIDAN MÜŞTERİ BİLGİLERİNİ ALMA
# ------------------------------------------------------------
# input() ile kullanıcıdan veri alıyoruz.
# float() fatura gibi ondalıklı sayılar için kullanılır.
# int() sadakat ayı gibi tam sayılar için kullanılır.

ad_soyad = input("Müşteri ad soyad giriniz: ")
aylik_ucret = float(input("Aylık fatura tutarını giriniz: "))
sadakat_ayi = int(input("Kaç aydır müşteri olduğunu giriniz: "))
aktif_girdi = input("Müşteri aktif mi? Evet/Hayır: ")
hizmet_girdi = input("Aldığı hizmetleri virgülle giriniz. Örnek: İnternet,TV,Mobil: ")


# ------------------------------------------------------------
# 3. GİRİLEN VERİLERİ DÜZENLEME
# ------------------------------------------------------------
# Kullanıcı 'Evet' yazarsa müşteri aktif kabul edilir.
# lower() sayesinde Evet, EVET, evet gibi yazımlar sorun çıkarmaz.

aktif_mi = aktif_girdi.lower() == "evet"

# Hizmetler virgülle ayrılarak listeye çevrilir.
# strip() baştaki ve sondaki boşlukları temizler.
# if h.strip() != "" kısmı boş hizmet yazılmasını engeller.
# Böylece çıktıdaki ['İnternet', 'Mobil', ''] gibi karmaşık görüntü oluşmaz.

hizmetler = []
for h in hizmet_girdi.split(","):
    temiz_hizmet = h.strip()
    if temiz_hizmet != "":
        hizmetler.append(temiz_hizmet)

# Her müşteri için rastgele bir müşteri numarası üretilir.
customer_id = "TELCO-" + str(random.randint(1000, 9999))


# ------------------------------------------------------------
# 4. MÜŞTERİ BİLGİLERİNİ SÖZLÜKTE TUTMA
# ------------------------------------------------------------
# Sözlük yapısı, müşteri bilgilerini düzenli saklamak için kullanılır.

musteri = {
    "id": customer_id,
    "ad_soyad": ad_soyad.title(),
    "aylik_ucret": aylik_ucret,
    "sadakat_ayi": sadakat_ayi,
    "aktif_mi": aktif_mi,
    "hizmetler": hizmetler
}


# ------------------------------------------------------------
# 5. FATURA HESAPLAMA FONKSİYONU
# ------------------------------------------------------------
# Eğer fatura 500 TL'den büyükse %10 indirim yapılır.
# math.ceil() ile sonuç yukarı yuvarlanır.

def fatura_hesapla(ucret):
    if ucret > 500:
        indirimli_ucret = ucret * 0.90
        return math.ceil(indirimli_ucret)
    else:
        return math.ceil(ucret)


# ------------------------------------------------------------
# 6. CHURN RİSKİ HESAPLAMA FONKSİYONU
# ------------------------------------------------------------
# Aşağıdaki durumlardan biri varsa müşteri riskli kabul edilir:
# 1. Müşteri aktif değilse
# 2. Müşteri 6 aydan az süredir aboneyse ve faturası yüksekse
# 3. Müşteri az hizmet alıyor ama faturası yüksekse

def churn_riski_hesapla(m):
    if not m["aktif_mi"]:
        return True
    elif m["sadakat_ayi"] < 6 and m["aylik_ucret"] > 500:
        return True
    elif len(m["hizmetler"]) <= 1 and m["aylik_ucret"] > 300:
        return True
    else:
        return False


# ------------------------------------------------------------
# 7. KAMPANYA ÖNERİSİ FONKSİYONU
# ------------------------------------------------------------
# Riskli müşteriye elde tutma kampanyası önerilir.
# Faturası yüksek müşteriye VIP kampanya önerilir.
# Diğer müşterilere standart memnuniyet kampanyası önerilir.

def kampanya_oner(m, risk):
    if risk:
        return "İndirim, müşteri araması ve özel sadakat kampanyası önerilir."
    elif m["aylik_ucret"] > 500:
        return "VIP müşteri kampanyası önerilir."
    else:
        return "Standart memnuniyet kampanyası önerilir."


# ------------------------------------------------------------
# 8. MÜŞTERİ TİPİNİ BELİRLEME
# ------------------------------------------------------------
# Faturası yüksek olan veya uzun süredir müşteri olan kişiler VIP kabul edilir.

if musteri["aylik_ucret"] > 500 or musteri["sadakat_ayi"] > 24:
    musteri["musteri_tipi"] = "VIP Müşteri"
else:
    musteri["musteri_tipi"] = "Standart Müşteri"


# ------------------------------------------------------------
# 9. HESAPLAMALARI SÖZLÜĞE EKLEME
# ------------------------------------------------------------

musteri["fatura_tarihi"] = str(date.today())
musteri["indirimli_fatura"] = fatura_hesapla(musteri["aylik_ucret"])
musteri["churn_riski"] = churn_riski_hesapla(musteri)
musteri["kampanya"] = kampanya_oner(musteri, musteri["churn_riski"])


# ------------------------------------------------------------
# 10. ÇIKTIYI DAHA ANLAŞILIR HALE GETİRME
# ------------------------------------------------------------
# Bu bölümde teknik ifadeleri kullanıcı dostu yazıya çeviriyoruz.
# True yerine 'Aktif' veya 'Risk Var' yazdırıyoruz.
# Listeyi köşeli parantezle değil, virgüllü normal metin olarak gösteriyoruz.

if musteri["aktif_mi"]:
    aktiflik_yazisi = "Aktif müşteri"
else:
    aktiflik_yazisi = "Pasif müşteri"

if musteri["churn_riski"]:
    risk_yazisi = "Risk var"
else:
    risk_yazisi = "Risk yok"

if len(hizmetler) > 0:
    hizmet_yazisi = ", ".join(hizmetler)
else:
    hizmet_yazisi = "Hizmet bilgisi girilmedi"

benzersiz_hizmetler = sorted(set(hizmetler))
if len(benzersiz_hizmetler) > 0:
    benzersiz_hizmet_yazisi = ", ".join(benzersiz_hizmetler)
else:
    benzersiz_hizmet_yazisi = "Hizmet bilgisi yok"


# ------------------------------------------------------------
# 11. SONUÇ RAPORUNU EKRANA YAZDIRMA
# ------------------------------------------------------------
# Çıktı sade ve okunabilir olması için düzenli cümlelerle yazdırılır.

print("\n" + "=" * 60)
print("MÜŞTERİ ANALİZ RAPORU")
print("=" * 60)
print("Müşteri Numarası      :", musteri["id"])
print("Müşteri Adı Soyadı    :", musteri["ad_soyad"])
print("Müşteri Tipi          :", musteri["musteri_tipi"])
print("Fatura Tarihi         :", musteri["fatura_tarihi"])
print("Normal Fatura         :", str(musteri["aylik_ucret"]) + " TL")
print("İndirimli Fatura      :", str(musteri["indirimli_fatura"]) + " TL")
print("Sadakat Süresi        :", str(musteri["sadakat_ayi"]) + " ay")
print("Aktiflik Durumu       :", aktiflik_yazisi)
print("Alınan Hizmetler      :", hizmet_yazisi)
print("Tekil Hizmet Listesi  :", benzersiz_hizmet_yazisi)
print("Churn Risk Durumu     :", risk_yazisi)
print("Kampanya Önerisi      :", musteri["kampanya"])
print("=" * 60)


# ------------------------------------------------------------
# 12. KISA YORUM BÖLÜMÜ
# ------------------------------------------------------------
# Bu bölüm, raporun sonunda sistemin neden böyle sonuç verdiğini açıklar.

print("\nKISA YORUM")
print("-" * 60)

if musteri["churn_riski"]:
    print("Bu müşteri için ayrılma riski tespit edilmiştir.")
    print("Bu nedenle müşteriye özel indirim veya sadakat kampanyası uygulanabilir.")
else:
    print("Bu müşteri için yüksek bir ayrılma riski tespit edilmemiştir.")
    print("Müşteri memnuniyetini artıracak standart kampanyalar uygulanabilir.")


# ------------------------------------------------------------
# 13. KRİTİK SORU CEVAPLARI
# ------------------------------------------------------------

print("\nKRİTİK SORU CEVAPLARI")
print("-" * 60)
print("1. Müşteri bilgileri neden sözlükte tutuldu?")
print("Çünkü sözlük yapısı; ad, fatura, sadakat süresi ve hizmetleri düzenli şekilde saklamayı sağlar.")

print("\n2. Churn riski nasıl hesaplandı?")
print("Müşteri pasifse, sadakati düşükse veya az hizmet almasına rağmen faturası yüksekse riskli kabul edilir.")
